[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q3/03_compare_and_interpret.ipynb)

# R1-Q3 Week 4 — Compare attributions to known stress biology

### This notebook produces a Strong / Mixed / Low Overlap verdict with the evidence underneath it.

This notebook compares the top-attributed genes from your SHAP analysis (Week 3) against curated lists of known stress-responsive genes — primarily Hakkak & Tohidfar 2026's consensus list. The result is the diagnostic for whether your classifier learned real stress biology or technical patterns in the data.

By the end of this notebook you will have:

- An overlap table comparing top-attributed genes against curated stress-responsive sets, with hypergeometric test results per stress class (`attribution_overlap.parquet`).
- The specific gene-level overlap detail showing which curated reference genes appeared in the top-N attributions per class (`overlap_genes.parquet`).
- An `attribution_comparison.json` recording the per-class overlap statistics, the test parameters, and the diagnostic verdict (Strong, Mixed, or Low Overlap), ready as the headline finding in your Week 4 paper and final presentation.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R1-Q3 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q3")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Load Notebook 02's outputs and cross-check the contracts

The first thing this notebook does is read everything it needs from upstream: your precommitments from Notebook 00, the SHAP attribution outputs from Notebook 02, and the classifier summary from Notebook 01. It then cross-checks them against each other so any drift between notebooks surfaces here rather than further down.

The cross-checks fall into three buckets:

1. **Precommit structure.** The four top-level keys are present, the data source is `"GEO"`, the class count is `8` in both the places it appears, and the `artifact_like_definition` block has the sub-keys later sections will read.
2. **Cross-notebook consistency.** The attribution method recorded in `attribution_summary.json` matches what the precommit asked for, and the classifier referenced by Notebook 02 is the one Notebook 01 wrote.
3. **Subset selection.** Notebook 03 only processes subsets whose classifier passed the accuracy gate in Notebook 01. The list of passing subsets is read from `classifier_summary.json`.

If anything is off, this section fails loudly with a message that names which file is the source of the problem and what the expected value was.

In [ ]:
import json

# --- Resolve paths -----------------------------------------------------
# OUTPUT_DIR is set up in the preamble via iri.output_dir().
precommit_path            = OUTPUT_DIR / "precommit.json"
attribution_scores_path   = OUTPUT_DIR / "attribution_scores.parquet"
top_attributed_genes_path = OUTPUT_DIR / "top_attributed_genes.json"
attribution_summary_path  = OUTPUT_DIR / "attribution_summary.json"
classifier_summary_path   = OUTPUT_DIR / "classifier_summary.json"

# --- Defensive load of each input --------------------------------------
def _load_required(path, loader, hint):
    try:
        return loader(path)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"\nCould not find {path.name}.\n"
            f"  Expected at: {path}\n\n"
            f"{hint}\n"
        ) from None

precommit            = _load_required(
    precommit_path, lambda p: json.loads(p.read_text()),
    "Re-run Notebook 00 end-to-end; its closeout writes precommit.json.",
)
attribution_scores   = _load_required(
    attribution_scores_path, pd.read_parquet,
    "Re-run Notebook 02 end-to-end; Section 6 writes attribution_scores.parquet.",
)
top_attributed_genes = _load_required(
    top_attributed_genes_path, lambda p: json.loads(p.read_text()),
    "Re-run Notebook 02 end-to-end; Section 6 writes top_attributed_genes.json.",
)
attribution_summary  = _load_required(
    attribution_summary_path, lambda p: json.loads(p.read_text()),
    "Re-run Notebook 02 end-to-end; Section 6 writes attribution_summary.json.",
)
classifier_summary   = _load_required(
    classifier_summary_path, lambda p: json.loads(p.read_text()),
    "Re-run Notebook 01 end-to-end; Section 4.3 writes classifier_summary.json.",
)

# --- Precommit: top-level keys -----------------------------------------
expected_top_keys = {
    "attribution_method",
    "reference_gene_set",
    "artifact_like_definition",
    "data_source_and_scope",
}
missing = expected_top_keys - set(precommit.keys())
assert not missing, (
    f"precommit.json is missing top-level keys: {sorted(missing)}.\n"
    f"Re-run Notebook 00 — its closeout writes the full schema."
)

# --- Precommit: data source and class count ----------------------------
data_scope = precommit["data_source_and_scope"]
assert data_scope["source"] == "GEO", (
    f"precommit.json: data_source_and_scope.source is "
    f"{data_scope['source']!r}; expected 'GEO'. The library serves GEO only."
)
assert data_scope["n_stress_classes"] == 8, (
    f"precommit.json: data_source_and_scope.n_stress_classes is "
    f"{data_scope['n_stress_classes']}; expected 8."
)

artifact_def = precommit["artifact_like_definition"]
assert artifact_def["n_classes"] == 8, (
    f"precommit.json: artifact_like_definition.n_classes is "
    f"{artifact_def['n_classes']}; expected 8."
)
assert artifact_def["n_classes"] == data_scope["n_stress_classes"], (
    "precommit.json: artifact_like_definition.n_classes does not match "
    "data_source_and_scope.n_stress_classes. Both should be 8."
)

# --- Precommit: artifact_like_definition sub-keys ----------------------
expected_artifact_subkeys = {
    "n_classes", "per_class", "overlap_clause", "metadata_clause",
    "combination", "rollup", "rationale",
}
missing_subkeys = expected_artifact_subkeys - set(artifact_def.keys())
assert not missing_subkeys, (
    f"precommit.json: artifact_like_definition is missing sub-keys: "
    f"{sorted(missing_subkeys)}. Re-run Notebook 00 Section 6."
)

# --- Cross-notebook: attribution method --------------------------------
precommit_method = precommit["attribution_method"]["method"]
nb02_method      = attribution_summary["setup"]["attribution_method"]
assert nb02_method == precommit_method, (
    f"Attribution method drift between notebooks:\n"
    f"  precommit.json:           {precommit_method!r}\n"
    f"  attribution_summary.json: {nb02_method!r}\n"
    f"Re-run Notebook 02 with the precommit-aligned method."
)

# --- Cross-notebook: classifier identity (best-effort) -----------------
# NB01 records classifier_type under metadata. If NB02 also recorded it
# under setup, cross-check; otherwise skip silently. The check is belt-
# and-suspenders, not a primary gate.
nb01_classifier_type = classifier_summary["metadata"]["classifier_type"]
nb02_classifier_type = attribution_summary["setup"].get("classifier_type")
if nb02_classifier_type is not None and nb02_classifier_type != nb01_classifier_type:
    raise AssertionError(
        f"Classifier identity drift between notebooks:\n"
        f"  classifier_summary.json:  {nb01_classifier_type!r}\n"
        f"  attribution_summary.json: {nb02_classifier_type!r}"
    )

# --- Subset selection: passing subsets only ----------------------------
gate = classifier_summary["accuracy_gate"]["per_subset"]
passing_subsets = sorted(
    name for name, info in gate.items() if info["passes_gate"]
)
all_subsets = sorted(gate.keys())
non_passing = sorted(set(all_subsets) - set(passing_subsets))

assert passing_subsets, (
    "No subsets passed the Notebook 01 accuracy gate. There is nothing "
    "to interpret here. Revisit Notebook 01's gate threshold or the "
    "classifier configuration."
)

# --- Summary print -----------------------------------------------------
print("Inputs loaded:")
for name, path, extra in [
    ("precommit.json",              precommit_path,            ""),
    ("attribution_scores.parquet",  attribution_scores_path,
        f"({attribution_scores.shape[0]} rows × {attribution_scores.shape[1]} cols)"),
    ("top_attributed_genes.json",   top_attributed_genes_path, ""),
    ("attribution_summary.json",    attribution_summary_path,  ""),
    ("classifier_summary.json",     classifier_summary_path,   ""),
]:
    size_kb = path.stat().st_size / 1e3
    print(f"  {name:<28}  {size_kb:>6.1f} KB  {extra}")
print()
print(f"Attribution method:   {precommit_method}")
print(f"Data source:          {data_scope['source']}, {data_scope['n_stress_classes']} classes")
print(f"Top-N per class:      {attribution_summary['top_attributions']['top_n']}")
print()
print(f"Subsets to process:   {passing_subsets}")
if non_passing:
    print(f"Subsets skipped:      {non_passing}  (did not pass NB01 accuracy gate)")

With the inputs loaded and the cross-checks passed, the rest of the notebook can trust that:

- The precommit's structure is intact and its values agree with Notebook 00's hardcoded facts (data source `GEO`, 8 stress classes).
- The attribution outputs in `attribution_scores`, `top_attributed_genes`, and `attribution_summary` come from a run whose method matches the precommit.
- The list of subsets to process is in `passing_subsets`. Subsets whose classifier didn't pass Notebook 01's accuracy gate are listed under `non_passing` for the EQ#1 writeup but are not analyzed further.

Section 2 builds the reference gene set from Hakkak's supplementary file 1 — the comparator the overlap clause will test against in Section 4.

### 2) Compare top-attributed genes against the precommitted reference set (Hakkak & Tohidfar 2026)

### 3) Test attribution-score correlation with technical metadata (batch, processing date, tissue)

### 4) Apply the precommitted "artifact-like" rule

### 5) Assign the outcome label (Strong / Mixed / Low Overlap)

### 6) Closeout: save the verdict and the evidence; finalize paper and presentation